# SpaceAmb — Prototype Demo

Architectural-semantic research prototype.  This notebook demonstrates the full pipeline:

1. Load atoms, programs, ambiances
2. Generate embeddings (with disk cache)
3. Score atoms against example queries
4. Generate descriptors via grammar-constrained similarity-weighted composition
5. Score descriptors
6. Export all outputs
7. Plots: heatmap · bar chart · PCA · (optional UMAP)

> **Methodological note:** Similarity scores reflect *semantic affinity* in
> embedding space — they are a research instrument, not empirical perceptual ground truth.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# When running from notebooks/, project root is one level up
ROOT = Path('.').resolve().parent
src_path = ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

## 1. Load Config and Datasets

In [ ]:
import yaml
from semantic_architecture.atoms import load_atoms, atoms_summary
from semantic_architecture.queries import (
    load_programs, load_ambiances, generate_all_queries
)

CONFIG_PATH = ROOT / 'config' / 'config.yaml'
with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

atoms     = load_atoms(ROOT / cfg['data']['atoms_path'])
programs  = load_programs(ROOT / cfg['data']['programs_path'])
ambiances = load_ambiances(ROOT / cfg['data']['ambiances_path'])

print(atoms_summary(atoms))
print(f"\nPrograms : {[p['text'] for p in programs]}")
print(f"Ambiances: {[a['text'] for a in ambiances]}")

## 2. Generate Embeddings (Cached)

In [ ]:
from semantic_architecture.embeddings import EmbeddingModel

emb_cfg = cfg['embedding']
model = EmbeddingModel(
    model_name=emb_cfg['model_name'],
    cache_dir=ROOT / emb_cfg['cache_dir'],
    batch_size=emb_cfg['batch_size'],
)

query_sets    = generate_all_queries(programs, ambiances)
space_qs      = query_sets['space']
ambiance_qs   = query_sets['ambiance']
combined_qs   = query_sets['combined']

atom_texts     = [a.text for a in atoms]
atom_families  = [a.family for a in atoms]
atom_ids       = [a.id for a in atoms]

# Use embedding_text (description-enriched) for semantic embedding;
# combined_text (short label) is preserved for display.
atom_embs      = model.load_or_compute(atom_texts,                                    'atoms')
space_embs     = model.load_or_compute([q.embedding_text for q in space_qs],          'queries_space')
ambiance_embs  = model.load_or_compute([q.embedding_text for q in ambiance_qs],       'queries_ambiance')
combined_embs  = model.load_or_compute([q.embedding_text for q in combined_qs],       'queries_combined')

print(f'Atom embedding shape    : {atom_embs.shape}')
print(f'Combined embedding shape: {combined_embs.shape}')
print(f'Queries: {len(space_qs)} space, {len(ambiance_qs)} ambiance, {len(combined_qs)} combined')

## 3. Score Atoms Against Example Queries

In [ ]:
from semantic_architecture.scoring import (
    ScoringWeights, score_items_against_queries
)
from semantic_architecture.analysis import (
    top_atoms_for_query, top_atoms_by_family, compare_item_across_queries
)

weights = ScoringWeights.from_config(cfg.get('scoring', {}))

atom_scores = score_items_against_queries(
    item_texts=atom_texts, item_families=atom_families, item_ids=atom_ids,
    item_embeddings=atom_embs,
    space_queries=space_qs, ambiance_queries=ambiance_qs,
    combined_queries=combined_qs,
    space_embeddings=space_embs, ambiance_embeddings=ambiance_embs,
    combined_embeddings=combined_embs,
    weights=weights,
)
print(f'Score table shape: {atom_scores.shape}')
atom_scores.head(3)

In [ ]:
from semantic_architecture.scoring import enrich_with_discriminative_scores

# Enrich with discriminative_score and zscore_score.
# discriminative_score = weighted_score − per-atom mean across all queries.
# This corrects for atoms that are broadly relevant to all room concepts
# (high mean similarity) and promotes atoms that are *specifically* relevant
# to this query.  See README §Discriminative Scoring for details.
atom_scores = enrich_with_discriminative_scores(atom_scores)
print("Score columns:", list(atom_scores.columns))
print(f"\nExample — mean discriminative score (should be ~0 per atom):")
print(atom_scores.groupby("item_id")["discriminative_score"].mean().describe().round(4))

In [ ]:
# Top 20 atoms for 'relaxing living room' — ranked by discriminative_score
q_text = 'relaxing living room'
top20 = top_atoms_for_query(ex_ids[q_text], atom_scores, k=20,
                             score_col='discriminative_score')
print(f'Top 20 atoms for "{q_text}" (discriminative ranking):')
top20

In [ ]:
# Top atoms by family for 'somber conference room' — discriminative ranking
q_text = 'somber conference room'
by_fam = top_atoms_by_family(ex_ids[q_text], atom_scores, k=5,
                               score_col='discriminative_score')
for fam, df in sorted(by_fam.items()):
    print(f'\n--- {fam} ---')
    print(df.to_string(index=False))

In [ ]:
# Top atoms by family for 'somber conference room'
q_text = 'somber conference room'
by_fam = top_atoms_by_family(ex_ids[q_text], atom_scores, k=5)
for fam, df in sorted(by_fam.items()):
    print(f'\n--- {fam} ---')
    print(df.to_string(index=False))

In [ ]:
# Cross-query: how does 'sofa' score across all 196 queries?
sofa_cross = compare_item_across_queries('sofa', atom_scores)
print("Top-10 queries for 'sofa':")
print(sofa_cross.head(10)[['combined_text', 'sim_space', 'sim_ambiance',
                            'sim_combined', 'weighted_score']].to_string(index=False))
print('\nBottom-5 queries:')
print(sofa_cross.tail(5)[['combined_text', 'weighted_score']].to_string(index=False))

## 4. Generate Descriptors via Grammar-Constrained Similarity-Weighted Composition

In [ ]:
from semantic_architecture.composition import generate_descriptors, descriptors_to_df

comp_cfg = cfg.get('composition', {})
descriptors = generate_descriptors(
    atoms=atoms,
    atom_embeddings=atom_embs,
    n_descriptors=comp_cfg.get('n_descriptors', 300),
    descriptor_lengths=comp_cfg.get('descriptor_lengths', [2, 3]),
    temperature=comp_cfg.get('temperature', 1.0),
    seed=comp_cfg.get('seed', 42),
)

desc_df = descriptors_to_df(descriptors)
print(f'Generated {len(descriptors)} unique descriptors')
print(f'\nFamily patterns (top 15):')
print(desc_df['family_pattern'].value_counts().head(15).to_string())
desc_df[['text', 'family_pattern', 'n_atoms']].head(25)

In [ ]:
# Show full provenance for first 5 descriptors
for d in descriptors[:5]:
    print(d.provenance_str())
    print()

## 5. Score Descriptors

In [ ]:
from semantic_architecture.analysis import top_descriptors_for_query

for q_text in EXAMPLE_QUERIES:
    if q_text not in ex_ids:
        continue
    top_d = top_descriptors_for_query(ex_ids[q_text], descriptor_scores, k=15,
                                       score_col='discriminative_score')
    print(f'\nTop 15 descriptors for "{q_text}" (discriminative ranking):')
    cols = ['rank', 'text', 'family', 'weighted_score', 'discriminative_score']
    print(top_d[[c for c in cols if c in top_d.columns]].to_string(index=False))

In [ ]:
desc_texts    = [d.text for d in descriptors]
desc_embs     = model.load_or_compute(desc_texts, 'descriptors')
desc_families = [d.source_atom_families[0] for d in descriptors]
desc_ids      = [d.id for d in descriptors]

descriptor_scores = score_items_against_queries(
    item_texts=desc_texts, item_families=desc_families, item_ids=desc_ids,
    item_embeddings=desc_embs,
    space_queries=space_qs, ambiance_queries=ambiance_qs,
    combined_queries=combined_qs,
    space_embeddings=space_embs, ambiance_embeddings=ambiance_embs,
    combined_embeddings=combined_embs,
    weights=weights,
)
print(f'Descriptor score table shape: {descriptor_scores.shape}')

In [ ]:
from semantic_architecture.analysis import top_descriptors_for_query

for q_text in EXAMPLE_QUERIES:
    if q_text not in ex_ids:
        continue
    top_d = top_descriptors_for_query(ex_ids[q_text], descriptor_scores, k=15)
    print(f'\nTop 15 descriptors for "{q_text}":')
    print(top_d[['rank', 'text', 'family', 'weighted_score']].to_string(index=False))

## 6. Export All Results

In [ ]:
from semantic_architecture.io_utils import save_csv, save_json

out_dir = ROOT / cfg['output']['dir']
out_dir.mkdir(parents=True, exist_ok=True)

save_csv(atom_scores,       out_dir / 'atom_scores.csv',       'atom scores')
save_csv(descriptor_scores, out_dir / 'descriptor_scores.csv', 'descriptor scores')
save_csv(desc_df,           out_dir / 'descriptors.csv',       'descriptor table')

# Per-query rankings
for q_text, qid in ex_ids.items():
    safe = q_text.replace(' ', '_')
    save_csv(
        top_atoms_for_query(qid, atom_scores, k=30),
        out_dir / f'ranking_atoms_{safe}.csv'
    )
    save_csv(
        top_descriptors_for_query(qid, descriptor_scores, k=30),
        out_dir / f'ranking_descriptors_{safe}.csv'
    )

print('Export complete.')

## 7. Visualizations

In [ ]:
# --- 7b. Bar chart: top 20 atoms for 'relaxing living room' (discriminative) ---
q_text = 'relaxing living room'
top20  = top_atoms_for_query(ex_ids[q_text], atom_scores, k=20,
                              score_col='discriminative_score')
fig = plot_top_items(
    top20, query_label=q_text, top_k=20,
    score_col='discriminative_score', color_by_family=True,
)
fig.savefig(out_dir / 'bar_atoms_relaxing_living_room.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 7a. Heatmap: top-25 atoms × 4 example queries ---
from semantic_architecture.scoring import similarity_matrix_df

# Gather embeddings for the 4 example combined queries
ex_combined_embs = []
ex_labels = []
for qt in EXAMPLE_QUERIES:
    idx = next(i for i, q in enumerate(combined_qs) if q.combined_text == qt)
    ex_combined_embs.append(combined_embs[idx])
    ex_labels.append(qt)
ex_combined_embs = np.stack(ex_combined_embs)

# Select top-25 atoms by mean weighted score across these 4 queries
ex_qids = list(ex_ids.values())
mean_scores = (
    atom_scores[atom_scores['query_id'].isin(ex_qids)]
    .groupby('text')['weighted_score'].mean()
    .sort_values(ascending=False)
    .head(25)
)
top25_texts = list(mean_scores.index)
top25_idx   = [atom_texts.index(t) for t in top25_texts]
top25_embs  = atom_embs[top25_idx]

mat = similarity_matrix_df(top25_texts, ex_labels, top25_embs, ex_combined_embs)
fig = plot_heatmap(mat, title='Top-25 atoms vs 4 example queries',
                   figsize=(10, 10), annot=True)
fig.savefig(out_dir / 'heatmap_atoms_4queries.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 7b. Bar chart: top 20 atoms for 'relaxing living room' ---
q_text = 'relaxing living room'
top20  = top_atoms_for_query(ex_ids[q_text], atom_scores, k=20)
fig = plot_top_items(
    top20, query_label=q_text, top_k=20,
    score_col='weighted_score', color_by_family=True,
)
fig.savefig(out_dir / 'bar_atoms_relaxing_living_room.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 7c. PCA of all atoms ---
fig = plot_pca_projection(
    atom_embs, atom_texts, families=atom_families,
    title='PCA of all atoms (coloured by family)',
    annotate=True, figsize=(12, 9),
)
fig.savefig(out_dir / 'pca_atoms.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 7d. UMAP (skipped if umap-learn not installed) ---
fig = plot_umap_projection(
    atom_embs, atom_texts, families=atom_families,
    title='UMAP of all atoms (coloured by family)',
    annotate=True, figsize=(12, 9),
)
if fig:
    fig.savefig(out_dir / 'umap_atoms.png', bbox_inches='tight', dpi=150)
    plt.show()

---

## 8. Phase 3 — Scene Descriptions

Scenes are paragraph-length architectural descriptions — the natural end product
of combining atoms and descriptors into fluent text.  The same embedding + cosine
similarity pipeline scores scenes against every space × ambiance query, making
them directly comparable to atoms and descriptors.

Each scene in `data/raw/scenes.json` carries an **intended query** (`space` ×
`ambiance`) that acts as ground truth for evaluation: does the pipeline rank a
scene highest for the query it was authored for?

In [ ]:
from semantic_architecture.scenes import load_scenes, scenes_summary, validate_scenes
import yaml

with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

scenes_path = ROOT / cfg['data']['scenes_path']
scenes = load_scenes(scenes_path)
print(scenes_summary(scenes))

# Validate against known programs/ambiances
valid_spaces = [p['text'] for p in programs]
valid_ambs   = [a['text'] for a in ambiances]
warnings = validate_scenes(scenes, valid_spaces, valid_ambs)
if warnings:
    for w in warnings:
        print(f'WARNING: {w}')
else:
    print('\nAll scenes valid.')

In [ ]:
# Embed scenes (cached independently from atoms and descriptors)
scene_texts    = [s.text for s in scenes]
scene_families = [s.space for s in scenes]   # use intended space as family label
scene_ids      = [s.id for s in scenes]

scene_embs = model.load_or_compute(scene_texts, 'scenes')
print(f'Scene embedding shape: {scene_embs.shape}')

In [ ]:
from semantic_architecture.scoring import score_items_against_queries, enrich_with_discriminative_scores
from semantic_architecture.analysis import top_scenes_for_query

scene_scores = score_items_against_queries(
    item_texts=scene_texts, item_families=scene_families, item_ids=scene_ids,
    item_embeddings=scene_embs,
    space_queries=space_qs, ambiance_queries=ambiance_qs,
    combined_queries=combined_qs,
    space_embeddings=space_embs, ambiance_embeddings=ambiance_embs,
    combined_embeddings=combined_embs,
    weights=weights,
)
scene_scores = enrich_with_discriminative_scores(scene_scores)
print(f'Scene score table: {scene_scores.shape}')

In [ ]:
# Top scenes for each of the 4 example queries
for q_text in EXAMPLE_QUERIES:
    if q_text not in ex_ids:
        continue
    top_s = top_scenes_for_query(ex_ids[q_text], scene_scores, k=5,
                                  score_col='discriminative_score')
    print(f'\nTop 5 scenes for "{q_text}":')
    print(top_s[['rank', 'text', 'family', 'discriminative_score']].to_string(index=False))

### 8a. Ground-Truth Evaluation

For each scene, check whether the pipeline's top-scoring query matches the
scene's intended `space × ambiance` target.  A hit means the scene is ranked
highest (or within the top-k) for the query it was authored for.  This is a
minimal sanity check — high-accuracy retrieval would suggest the embedding space
reliably separates space × ambiance combinations.

In [ ]:
results = []
for scene in scenes:
    intended = scene.intended_query    # e.g. "relaxing living room"
    # Get this scene's score across all queries, sorted by discriminative_score
    scene_row = scene_scores[scene_scores['item_id'] == scene.id].copy()
    scene_row = scene_row.sort_values('discriminative_score', ascending=False)

    top1_text  = scene_row.iloc[0]['combined_text']
    top1_score = scene_row.iloc[0]['discriminative_score']

    # Rank of the intended query for this scene
    intended_rows = scene_row[scene_row['combined_text'] == intended]
    rank_of_intended = (
        int(scene_row.index.get_loc(intended_rows.index[0])) + 1
        if len(intended_rows) > 0 else None
    )

    results.append({
        'scene_id': scene.id,
        'intended': intended,
        'top1_query': top1_text,
        'hit@1': top1_text == intended,
        'rank_of_intended': rank_of_intended,
    })

eval_df = pd.DataFrame(results)
hit1 = eval_df['hit@1'].mean()
hit5 = (eval_df['rank_of_intended'] <= 5).mean()
print(f'Hit@1  : {hit1:.0%}  ({eval_df["hit@1"].sum()}/{len(eval_df)} scenes)')
print(f'Hit@5  : {hit5:.0%}  ({(eval_df["rank_of_intended"] <= 5).sum()}/{len(eval_df)} scenes)')
print()
eval_df[['scene_id', 'intended', 'top1_query', 'hit@1', 'rank_of_intended']]

In [ ]:
# Export scene scores and evaluation
save_csv(scene_scores, out_dir / 'scene_scores.csv', 'scene scores')
save_csv(eval_df, out_dir / 'scene_eval.csv', 'scene ground-truth evaluation')
print('Scene outputs exported.')

In [ ]:
# --- 7e. Family comparison across 4 queries ---
fig = plot_family_comparison(
    atom_scores, query_ids=ex_qids,
    score_col='weighted_score', figsize=(14, 6),
)
fig.savefig(out_dir / 'family_comparison_4queries.png', bbox_inches='tight', dpi=150)
plt.show()

## Summary

All outputs have been saved to `data/processed/`:

| File | Contents |
|------|----------|
| `atom_scores.csv` | Full scoring table (atoms × all 196 queries) |
| `descriptor_scores.csv` | Full scoring table (descriptors × all 196 queries) |
| `descriptors.csv` | Descriptor list with provenance |
| `ranking_atoms_*.csv` | Per-query atom rankings |
| `ranking_descriptors_*.csv` | Per-query descriptor rankings |
| `heatmap_atoms_4queries.png` | Similarity heatmap |
| `bar_atoms_relaxing_living_room.png` | Bar chart |
| `pca_atoms.png` | PCA projection |

> **Next steps:** Adjust `config/config.yaml` to change the embedding model,
> composition temperature, or scoring weights, then re-run this notebook.